# 🚀 Goal-Based Agentic AI: Dynamic Route Planning System

This notebook implements:
- CrewAI agent system
- Groq LLM (Llama 3.1 8B Instant)
- Simple Reflex Agent (Safety Shutdown)
- Dynamic Route Planning (Weather + Traffic)
- Gradio UI (Professional Light Theme)


In [ ]:
# 1. Install required libraries
!pip install crewai gradio groq

In [ ]:
# 2. Import libraries
from crewai import Agent, Task, Crew
from groq import Groq
import gradio as gr

In [ ]:
# 3. Setup Groq API Key
GROQ_API_KEY = "YOUR_GROQ_API_KEY"
client = Groq(api_key=GROQ_API_KEY)

In [ ]:
# 4. Simple Reflex Agent (Safety Shutdown)
def safety_check(engine_temp, threshold=100):
    if engine_temp > threshold:
        return "🚨 CRITICAL: Engine Overheated! Initiating Automatic Shutdown..."
    return "✅ Engine Status Normal"

In [ ]:
# 5. Route Planning Logic
def route_planner(weather, traffic):
    if weather == "bad" or traffic == "high":
        return "Alternative Safe Route Selected"
    return "Optimal Fastest Route Selected"

In [ ]:
# 6. LLM Decorated Output using Groq
def generate_output(route, safety):
    prompt = f"""
    You are an aerospace AI assistant.
    Format the following output professionally:
    Route Decision: {route}
    Safety Status: {safety}
    """
    
    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [ ]:
# 7. CrewAI Agent Setup
planner_agent = Agent(
    role="Route Planner",
    goal="Determine optimal route",
    backstory="Expert in navigation and optimization"
)

task = Task(
    description="Plan route based on weather and traffic",
    agent=planner_agent
)

crew = Crew(agents=[planner_agent], tasks=[task])

In [ ]:
# 8. Full System Function
def run_system(weather, traffic, engine_temp):
    route = route_planner(weather, traffic)
    safety = safety_check(engine_temp)
    return generate_output(route, safety)

In [ ]:
# 9. Gradio UI (Professional Light Theme)
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ✈️ AI Route Planning System")
    
    weather = gr.Dropdown(["good", "bad"], label="Weather Condition")
    traffic = gr.Dropdown(["low", "high"], label="Traffic Level")
    engine_temp = gr.Slider(0, 150, label="Engine Temperature")
    
    output = gr.Textbox(label="AI Decision Output")
    
    btn = gr.Button("Run System")
    btn.click(run_system, inputs=[weather, traffic, engine_temp], outputs=output)

demo.launch()